In [1]:
%%file config.py

KAFKA_SERVER = "broker:9092"
TOPIC = "event-weather-stream"

EVENTS = [
    {
        "event_id": "WAW-CONCERT-001",
        "city": "Warszawa",
        "venue": "PGE Narodowy - outdoor stage",
    },
    {
        "event_id": "WAW-FEST-002",
        "city": "Warszawa",
        "venue": "Old Town Square Festival Stage",
    }
]

Writing config.py


In [10]:
%%file weather_gen.py

import random
from datetime import datetime

def generate_weather(event):

    wind = random.randint(10, 55)
    gust = wind + random.randint(0, 25)
    rain = round(random.uniform(0, 6), 2)
    visibility = random.randint(3000, 10000)

    # extreme weather injection (5%)
    if random.random() < 0.05:
        wind = random.randint(70, 130)
        gust = wind + random.randint(10, 40)
        rain = round(random.uniform(8, 25), 2)
        visibility = random.randint(500, 3000)

    return {
        "event_id": event["event_id"],
        "city": event["city"],
        "venue": event["venue"],
        "wind_kmh": wind,
        "gust_kmh": gust,
        "rain_intensity": rain,
        "visibility_m": visibility,
        "timestamp": datetime.utcnow().isoformat(),
        "source": "event_weather_simulator"
    }

Writing weather_gen.py


In [9]:
%%file producer.py

import json
import time
from datetime import datetime
from kafka import KafkaProducer

from config import KAFKA_SERVER, TOPIC, EVENTS
from weather_gen import generate_weather

producer = KafkaProducer(
    bootstrap_servers=KAFKA_SERVER,
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

event_index = 0

print("Producer started...")

while True:

    event = EVENTS[event_index % len(EVENTS)]
    event_index += 1

    weather_event = generate_weather(event)

    payload = {
        **weather_event,
        "ingested_at": datetime.utcnow().isoformat()
    }

    producer.send(TOPIC, payload)

    print(
        f"{payload['event_id']} | "
        f"city={payload['city']} | "
        f"wind={payload['wind_kmh']} km/h | "
        f"gust={payload['gust_kmh']} km/h | "
        f"rain={payload['rain_intensity']} mm | "
        f"visibility={payload['visibility_m']} m"
    )

    time.sleep(1.2)

Overwriting producer.py


In [11]:
%%file risk_engine.py

import json
from kafka import KafkaConsumer, KafkaProducer

from config import KAFKA_SERVER, TOPIC

consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=KAFKA_SERVER,
    auto_offset_reset="latest",
    value_deserializer=lambda x: json.loads(x.decode("utf-8")),
    group_id="risk-engine-v2"
)


alert_producer = KafkaProducer(
    bootstrap_servers=KAFKA_SERVER,
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)


def compute_risk(event):

    score = 0
    reasons = []

    wind = event["wind_kmh"]
    gust = event["gust_kmh"]
    rain = event["rain_intensity"]
    visibility = event["visibility_m"]

    # wind risk
    if wind > 60:
        score += 3
        reasons.append("Strong wind > 60 km/h")

    if wind > 90:
        score += 2
        reasons.append("Extreme wind > 90 km/h")

    # gust risk
    if gust > 80:
        score += 2
        reasons.append("High wind gust > 80 km/h")

    # rain risk
    if rain > 10:
        score += 2
        reasons.append("Heavy rain > 10 mm")

    # visibility risk
    if visibility < 2000:
        score += 2
        reasons.append("Low visibility < 2000m")

    # final decision
    if score >= 7:
        level = "CRITICAL"
    elif score >= 4:
        level = "HIGH"
    elif score >= 2:
        level = "MEDIUM"
    else:
        level = "LOW"

    return score, level, reasons


print("Risk Engine started...")

for msg in consumer:

    event = msg.value

    score, level, reasons = compute_risk(event)

    decision = {
        "event_id": event["event_id"],
        "city": event["city"],
        "venue": event["venue"],
        "risk_score": score,
        "risk_level": level,
        "reasons": reasons,
        "wind": event["wind_kmh"],
        "gust": event["gust_kmh"],
        "rain": event["rain_intensity"],
        "visibility": event["visibility_m"],
        "timestamp": event["timestamp"],
        "ingested_at": event["ingested_at"]
    }

    print(
        f"{decision['event_id']} | "
        f"RISK={level} ({score}) | "
        f"wind={event['wind_kmh']} gust={event['gust_kmh']}"
    )

    if level in ["HIGH", "CRITICAL"]:

        alert_producer.send("event-alerts", decision)

        print("ALERT SENT ->", level)

Overwriting risk_engine.py


In [5]:
%%file alert_consumer.py

import json
from kafka import KafkaConsumer

from config import KAFKA_SERVER

consumer = KafkaConsumer(
    "event-alerts",
    bootstrap_servers=KAFKA_SERVER,
    auto_offset_reset="latest",
    value_deserializer=lambda x: json.loads(x.decode("utf-8")),
    group_id="alert-dashboard"
)

alert_count = 0
critical_count = 0
high_count = 0

latest_alerts = [] 
MAX_BUFFER = 5


def print_dashboard():
    print("\n" + "=" * 70)
    print("EVENT WEATHER LIVE DASHBOARD")
    print("=" * 70)

    print(f"Total alerts   : {alert_count}")
    print(f"HIGH alerts    : {high_count}")
    print(f"CRITICAL alerts: {critical_count}")

    print("\nLatest alerts:")

    for a in latest_alerts[-MAX_BUFFER:]:
        print(
            f"- {a['event_id']} | {a['city']} | "
            f"RISK={a['risk_level']} ({a['risk_score']}) | "
            f"wind={a['wind']} gust={a['gust']} rain={a['rain']} visibility={a['visibility']}"
        )

    print("=" * 70 + "\n")


print("Weather Alert Dashboard started... waiting for events\n")

for msg in consumer:

    alert = msg.value

    alert_count += 1
    latest_alerts.append(alert)

    if alert["risk_level"] == "CRITICAL":
        critical_count += 1
    elif alert["risk_level"] == "HIGH":
        high_count += 1

    # live refresh dashboard
    print_dashboard()

Overwriting alert_consumer.py


In [3]:
%%file alert_consumer_detailed.py

import json
from kafka import KafkaConsumer
from collections import deque

from config import KAFKA_SERVER


consumer = KafkaConsumer(
    "event-alerts",
    bootstrap_servers=KAFKA_SERVER,
    auto_offset_reset="latest",
    value_deserializer=lambda x: json.loads(x.decode("utf-8")),
    group_id="alert-dashboard-v2"
)

# ----------------------------
# STATE
# ----------------------------
alert_count = 0
critical_count = 0
high_count = 0

WINDOW_SIZE = 20
window = deque(maxlen=WINDOW_SIZE)


# ----------------------------
# STATS HELPERS
# ----------------------------
def calc_avg(field):
    if not window:
        return 0
    return sum(a[field] for a in window) / len(window)


def print_dashboard():
    print("\n" + "=" * 80)
    print("EVENT WEATHER LIVE DASHBOARD")
    print("=" * 80)

    print(f"Total alerts   : {alert_count}")
    print(f"HIGH alerts    : {high_count}")
    print(f"CRITICAL alerts: {critical_count}")

    print("\nLATEST ALERTS:")

    for a in list(window)[-5:]:
        print("\n----------------------------------------")
        print(f"Event      : {a['event_id']}")
        print(f"City       : {a['city']}")
        print(f"Venue      : {a['venue']}")
        print(f"Risk level : {a['risk_level']} ({a['risk_score']})")
        print(f"Time       : {a['timestamp']}")
        print(f"Ingested   : {a['ingested_at']}")

        print("\nWeather snapshot:")
        print(f"  wind       : {a['wind']} km/h")
        print(f"  gust       : {a['gust']} km/h")
        print(f"  rain       : {a['rain']} mm")
        print(f"  visibility : {a['visibility']} m")

        print("\nReasons:")
        for r in a.get("reasons", []):
            print(f"  - {r}")

    print("\n" + "=" * 80 + "\n")


print("Weather Alert Dashboard started...\n")

for msg in consumer:

    alert = msg.value

    alert_count += 1
    window.append(alert)

    if alert["risk_level"] == "CRITICAL":
        critical_count += 1
    elif alert["risk_level"] == "HIGH":
        high_count += 1

    print_dashboard()

Writing alert_consumer_detailed.py
